# 12.2: Detecting contamination: n-gram overlap

An example is flagged as contaminated if the overlap exceeds a threshold (for instance, any single long n-gram match, or more than half of its n-grams). The choice of $n$ is a trade-off. Small $n$ (say 5) produces false positives, because common phrases like "which of the following is" appear everywhere. Large $n$ (say 50) produces false negatives, because a paraphrase or a change in formatting breaks every match.

Here is a minimal implementation. Real systems use hashing, Bloom filters, or suffix arrays to handle trillion-token corpora, but the logic is the same.


In [ ]:
import re

def ngrams(text: str, n: int) -> set[tuple[str, ...]]:
    words = re.findall(r"\w+", text.lower())
    return {tuple(words[i:i + n]) for i in range(len(words) - n + 1)}

def build_index(corpus_docs, n: int) -> set[tuple[str, ...]]:
    index = set()
    for doc in corpus_docs:
        index |= ngrams(doc, n)
    return index

def overlap_score(example: str, index: set, n: int) -> float:
    grams = ngrams(example, n)
    if not grams:
        return 0.0
    return len(grams & index) / len(grams)

corpus = ["Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May."]
index = build_index(corpus, n=8)
test_item = "Natalia sold clips to 48 of her friends in April, and then she sold half as many in May. How many clips did she sell?"
print(round(overlap_score(test_item, index, n=8), 2))   # a high score flags likely contamination
